# HCLTech Career Shaper - Final Academic Project
## Project 10: Employee Attrition Analysis
**Candidate Name:** Ranjan Maiti  
**Candidate Roll No:** 2500321540146  
**Domain:** HR & Workforce Analytics  
**Tech Stack:** Python (pandas, numpy, seaborn, matplotlib), Oracle SQL, Power BI
---
### Project Objectives
1. Ingest and clean the benchmark IBM HR Analytics Employee Attrition dataset (1,470 employees).
2. Identify top attrition drivers across Departments, Tenure bands, Salary bands, and OverTime status.
3. Conduct statistical outlier detection on compensation (IQR method) and validate boundary cases.
4. Build dimension and fact tables following star-schema enterprise architecture for Oracle SQL and Power BI ingestion.
5. Reconcile KPIs across Python, Oracle SQL, and Power BI.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = 'Arial'
print('Libraries imported successfully!')

### Step 1: Data Ingestion & Integrity Check

In [2]:
raw_data_path = os.path.join('..', 'data', 'raw', 'WA_Fn-UseC_-HR-Employee-Attrition.csv')
df_raw = pd.read_csv(raw_data_path)
print(f'Dataset Dimensions: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns')
print(f'Total Null Values: {df_raw.isnull().sum().sum()}')
print(f'Total Duplicate Rows: {df_raw.duplicated().sum()}')
df_raw.head(3)

### Step 2: Data Cleaning & Feature Engineering
- Drop zero-variance columns (`EmployeeCount`, `Over18`, `StandardHours`)
- Create numeric `Attrition_Flag` (1 = Yes, 0 = No)
- Discretize `SalaryBand` (<3.5k, 3.5k-7k, 7k-12k, 12k+)
- Discretize `TenureBand` (0-2 Yrs, 3-5 Yrs, 6-10 Yrs, 10+ Yrs)

In [3]:
df_clean = df_raw.copy()
# Strip BOM / whitespaces
df_clean.columns = [c.strip().replace('\ufeff', '') for c in df_clean.columns]

# Drop constant columns
constant_cols = ['EmployeeCount', 'Over18', 'StandardHours']
df_clean.drop(columns=constant_cols, inplace=True, errors='ignore')

# Binary flag
df_clean['Attrition_Flag'] = df_clean['Attrition'].apply(lambda x: 1 if str(x).strip().lower() == 'yes' else 0)

# Salary Bins
bins_salary = [0, 3500, 7000, 12000, 100000]
labels_salary = ['Low (<3.5k)', 'Mid (3.5k-7k)', 'High (7k-12k)', 'Executive (12k+)']
df_clean['SalaryBand'] = pd.cut(df_clean['MonthlyIncome'], bins=bins_salary, labels=labels_salary, right=False)

# Tenure Bins
bins_tenure = [0, 3, 6, 11, 100]
labels_tenure = ['0-2 Years', '3-5 Years', '6-10 Years', '10+ Years']
df_clean['TenureBand'] = pd.cut(df_clean['YearsAtCompany'], bins=bins_tenure, labels=labels_tenure, right=False)

print('Engineered features created successfully.')
print(f'Cleaned dataset shape: {df_clean.shape}')

### Step 3: Outlier Detection on Salary (IQR Method)

In [4]:
q1 = df_clean['MonthlyIncome'].quantile(0.25)
q3 = df_clean['MonthlyIncome'].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
salary_outliers = df_clean[df_clean['MonthlyIncome'] > upper_bound]

print(f'Q1: ${q1:.2f}, Q3: ${q3:.2f}, IQR: ${iqr:.2f}')
print(f'Upper threshold for salary outliers: ${upper_bound:.2f}')
print(f'Detected Outlier Count: {len(salary_outliers)} ({len(salary_outliers)/len(df_clean)*100:.2f}%)')
print('\nOutliers breakdown by Job Role:')
print(salary_outliers['JobRole'].value_counts())

### Step 4: Key KPI Summary & Exploratory Analysis

In [5]:
total_emp = len(df_clean)
attrition_cnt = df_clean['Attrition_Flag'].sum()
attrition_rate = (attrition_cnt / total_emp) * 100
print(f'Total Employees: {total_emp}')
print(f'Active Employees: {total_emp - attrition_cnt}')
print(f'Attrition Count: {attrition_cnt}')
print(f'Overall Attrition Rate: {attrition_rate:.2f}%')

print('\n--- Attrition by Department ---')
dept_summary = df_clean.groupby('Department').agg(
    Total=('EmployeeNumber', 'count'),
    Attrition=('Attrition_Flag', 'sum'),
    Rate_Pct=('Attrition_Flag', lambda x: x.mean() * 100)
)
display(dept_summary)

print('\n--- Attrition by Tenure Band ---')
tenure_summary = df_clean.groupby('TenureBand', observed=False).agg(
    Total=('EmployeeNumber', 'count'),
    Attrition=('Attrition_Flag', 'sum'),
    Rate_Pct=('Attrition_Flag', lambda x: x.mean() * 100)
)
display(tenure_summary)

### Step 5: Visual Insights and Driver Analysis

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Attrition by Department
sns.barplot(data=df_clean, x='Department', y='Attrition_Flag', ax=axes[0], palette='Blues_d', errorbar=None)
axes[0].set_title('Attrition Rate by Department', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Attrition Rate')

# Attrition by OverTime
sns.barplot(data=df_clean, x='OverTime', y='Attrition_Flag', ax=axes[1], palette='Reds_d', errorbar=None)
axes[1].set_title('Attrition Rate by OverTime Requirement', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Attrition Rate')
plt.tight_layout()
plt.show()

### Step 6: Conclusion & Business Recommendations
1. **Early Career Flight Risk:** Employees in their first 2 years exhibit nearly 30% attrition. A 90-day onboarding mentorship is strongly recommended.
2. **OverTime Burnout:** Overtime workers experience 30.5% attrition vs 10.4% for non-overtime. Workload rebalancing is urgent.
3. **Compensation Parity:** Low-salary tier has 27.1% attrition. Entry-level compensation benchmark reviews are required.